# 01 · Explore the OpenAlex ALAN papers

Matches the title and abstract of each paper against the word lists in `vocab/terms.py`, flags papers that are off-topic for the biodiversity focus (human health), marks duplicate and peer-review records, links matched words to knowledge-graph items, and builds a small local graph (toy model) that answers two questions from `questions.md`.

**Reads:** `data/raw/openalex_alan.csv` (made by `src/fetch_openalex.py`), `kg/seed_statements.csv`, `kg/items.csv`, `kg/term_map.csv`
**Writes:** `data/output/table_abstracts.csv` and `data/output/mentions.csv` (both git-ignored)

**After editing `vocab/terms.py` or `kg/term_map.csv`:** click Restart, then Run All. Python keeps the old word lists in memory otherwise.

## Progress log

| Date | Change | Papers with no match | In-scope papers with no match |
|---|---|---|---|
| 17 Nov | Round 1 baseline | 949 of 2,227 | not measured yet |
| 17 Nov | + zebrafish, elasmobranch, shark | 938 | not measured yet |
| 17 Nov | + seabird, petrel, Procellariiformes | 937 | 452 of 1,576 |
| 18 Nov | + Drosophila, deer, mouse, mice, rat, lizard, anole, frog, amphipod, chipmunk, epiphyte, wildlife | 804 | 385 of 1,666 |
| 19 Nov | + animals (mosquito, cricket, mussel, insect, ...), sky and policy words, first extension of the health list | 637 | 274 of 1,632 |
| 19 Nov | Health list pruned (words that also apply to animals removed), seed-item terms added (luminance, cd/m2, ...) | 614 | 277 of 1,668 |

3-way scope after the 19 Nov changes: in scope 1,668 · off-topic 455 · unsure 104.

Add a row here after each round of word-list changes.

## 1. Setup: load the papers and define the matcher

In [1]:
import re
import sys
import pandas as pd

sys.path.append("..")
from vocab import terms

df = pd.read_csv("../data/raw/openalex_alan.csv")
df["text"] = df["title"].fillna("") + ". " + df["abstract"].fillna("")

`make_pattern` builds a whole-word regex for each term. Abbreviations (SQM, PAR, lx) must match exactly. Other terms ignore case and also match a simple plural ("moth" matches "moths").

A term has to start at a word boundary, so "bird" does not match inside "seabird" and "fish" does not match inside "zebrafish": list compounds as their own terms. Irregular plurals need their own entry too ("mouse" and "mice").

In [2]:
def make_pattern(term):
    # Abbreviations like SQM, PAR, W must match exactly (capital letters).
    case_sensitive = term.isupper() or term == "lx"
    if case_sensitive:
        return re.compile(r"(?<!\w)" + re.escape(term) + r"(?!\w)")
    # Other terms ignore case and also match a plural ("moth" -> "moths").
    return re.compile(r"(?<!\w)" + re.escape(term) + r"(?:e?s)?(?!\w)", re.IGNORECASE)


def find_terms(text, term_list):
    found = [term for term in term_list if make_pattern(term).search(text)]
    return "; ".join(found)

## 2. Round 1 matching

One new column per category holds the terms found in each paper's title and abstract. `no_match` is True when a paper matches nothing in any of the five categories.

In [3]:
categories = {
    "methods":     terms.METHODS_INSTRUMENTS,
    "quantities":  terms.QUANTITIES,
    "units":       terms.UNITS,
    "disciplines": terms.DISCIPLINES,
    "species":     terms.TAXA,
}

# One new column per category: the terms found in each paper
for column, term_list in categories.items():
    df[column] = df["text"].apply(lambda t: find_terms(t, term_list))

# True for papers where nothing matched in any of the five categories
no_match = (df[list(categories)] == "").all(axis=1)
print(no_match.sum(), "of", len(df), "papers have no match")

614 of 2227 papers have no match


## 3. Scope: is the paper about biodiversity?

The project follows the Hölker et al. 2021 "11 pressing research questions" paper (biodiversity), so human-health papers are off-topic, unless they are about animals.

**Simple flag (first cell below):** human-health word AND no species match. A flagged paper can still match another category (unit, quantity), so flagged and unmatched overlap.

**3-way `scope` (second cell, refines the simple flag):**
- `off-topic`: flagged, with 2 or more health terms or one strong term (cancer, patient, diabetes, ...)
- `unsure`: flagged on a single weak term (sleep, participant, clinical, ...)
- `in scope`: everything else

In [4]:
df["human_health"] = df["text"].apply(lambda t: find_terms(t, terms.HUMAN_HEALTH))

off_topic = (df["human_health"] != "") & (df["species"] == "")
in_scope = ~off_topic

print(off_topic.sum(), "papers flagged off-topic (human health, no species)")
print((no_match & in_scope).sum(), "of", in_scope.sum(), "in-scope papers have no match")

559 papers flagged off-topic (human health, no species)
277 of 1668 in-scope papers have no match


In [5]:
strong = ["cancer", "cardiovascular", "obesity", "diabetes", "patient",
          "COVID", "depression", "mental health", "insomnia", "epidemiology"]

n_terms = df["human_health"].apply(lambda s: len(s.split("; ")) if s else 0)
strong_hits = df["human_health"].apply(
    lambda s: sum(t in strong for t in s.split("; ")))

flagged = (df["human_health"] != "") & (df["species"] == "")

df["scope"] = "in scope"
df.loc[flagged & ((n_terms >= 2) | (strong_hits >= 1)), "scope"] = "off-topic"
df.loc[flagged & (n_terms == 1) & (strong_hits == 0), "scope"] = "unsure"

df["scope"].value_counts()

scope
in scope     1668
off-topic     455
unsure        104
Name: count, dtype: int64

## 4. Summary numbers

In scope = all papers minus flagged. The flagged group and the no-match group overlap, so the table splits each of them.

In [6]:
n = len(df)
matched = ~no_match

summary = pd.DataFrame({
    "group": [
        "All papers",
        "Match at least one category",
        "Match nothing",
        "Flagged off-topic (human health, no species)",
        "   flagged AND match nothing",
        "   flagged BUT match something else",
        "In scope (not flagged)",
        "   in scope AND match nothing",
        "   in scope AND match something",
    ],
    "papers": [
        n,
        matched.sum(),
        no_match.sum(),
        off_topic.sum(),
        (off_topic & no_match).sum(),
        (off_topic & matched).sum(),
        in_scope.sum(),
        (in_scope & no_match).sum(),
        (in_scope & matched).sum(),
    ],
})
summary["% of all"] = (summary["papers"] / n * 100).round(1)
summary

,group,papers,% of all
0,All papers,2227,100.0
1,Match at least one category,1613,72.4
2,Match nothing,614,27.6
3,"Flagged off-topic (human health, no species)",559,25.1
4,flagged AND match nothing,337,15.1
5,flagged BUT match something else,222,10.0
6,In scope (not flagged),1668,74.9
7,in scope AND match nothing,277,12.4
8,in scope AND match something,1391,62.5


## 5. Checking the flag

Reading samples to see whether the flag removes papers it should not. Look for animals, ecology and light-measurement papers.

In [7]:
df[off_topic][["title", "human_health"]].sample(15, random_state=2)

,title,human_health
2205,PDE10A as a novel diagnostic and therapeutic t...,cancer
1967,Non-Image-Forming Effects of Daytime Electric ...,sleep
652,0169 Caffeine and Bright Light Effects on Mood...,participant; sleep; bedtime
549,Ligustilide alleviates the mitophagy and anxie...,anxiety
877,Independent and combined relationships between...,participant; cohort; cardiovascular
907,THE ROLE OF NURSING MANAGEMENT IN OPTIMIZING T...,patient; participant; insomnia; sleep
1098,Nighttime lighting effects on perceived restor...,participant
573,"Non-chemical, chemical, and biochemical, endoc...",clinical; neonatal
899,Chronotype and Mental Health: Are Late Sleeper...,sleep; mental health
685,"Alertness, arousal and anxiety in the spotligh...",participant; anxiety


In [8]:
n_terms = df["human_health"].apply(lambda s: len(s.split("; ")) if s else 0)
one_term = off_topic & (n_terms == 1)

print(one_term.sum(), "of", off_topic.sum(), "flagged on a single term")
df[one_term]["human_health"].value_counts()

151 of 559 flagged on a single term


human_health
sleep             35
participant       28
cancer            14
clinical          12
cardiovascular    11
cohort             6
diabetes           4
anxiety            4
patient            4
obesity            4
COVID              3
glucose            3
wearable           3
women              3
children           3
depression         2
childhood          2
epidemiology       2
mental health      2
adolescent         2
neonatal           1
insomnia           1
ergonomic          1
shift work         1
Name: count, dtype: int64

In [9]:
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)

unsure = df[df["scope"] == "unsure"][["title", "human_health"]]
unsure.sort_values("human_health")

,title,human_health
270,Long-term outdoor artificial light at night and psychological symptoms among Chinese adolescents...,adolescent
1155,"The independent and joint associations of community space environmental satisfaction, and parent...",adolescent
1112,The Light–Eye–Brain Axis: Neurobiological Links Between Mood Disorders and Myopia—A Narrative Re...,anxiety
1537,Pedestrians’ alertness and perceived environmental safety under non-uniform urban lighting,anxiety
90,Association between artificial light at night and the incidence of anxiety: a global ecological ...,anxiety
549,Ligustilide alleviates the mitophagy and anxiety-like behavior produced by nighttime lighting,anxiety
1852,Appropriate Lifelong Circadian Rhythms Are Established During Infancy: A Narrative Review,childhood
464,Reclaiming the Right to Starlight: A Human Rights Perspective on Light Pollution and Urban Crimi...,childhood
168,Reducing artificial light at night exposure in the urban environment could decrease light pollut...,children
1070,Measuring the visual environment of children and young people at risk of myopia: a scoping review,children


**Findings from reading the unsure list (18 Nov, when it had 151 papers):**
- About 23 wildlife and ecology papers were hidden only because their animal was missing from `TAXA` (fixed by adding animal terms).
- About 14 policy, regulation and governance papers (dark-sky governance, lighting standards, public perception): possibly the "units to legal issues" thread.
- A few light-measurement papers (nighttime Earth observation, city light pollution quantification, sensors).
- Some retrieved papers never mention light (equestrian business, malaria, photography): a "mentions light" flag would catch them.

**Decisions on `HUMAN_HEALTH` (19 Nov):**
- A health word should signal humans, human subjects or human settings. Words for biology that animals share (neurodegenerative, nephrotoxicity, endocrine disruptor, pregnancy, chronotype, maternal, retinal, oxidative stress, cognitive) were left out or removed.
- `cancer` stays. Of 17 in-scope papers that mention it, about 14 were human health. The animal ones (a rat model, spiders) are protected by their species words.
- The animal exception works both ways: a human paper that mentions an animal word in passing (for example "wildlife" or "mouse") is kept in scope, so samples still need a hand check.

## 6. Mark duplicates and peer-review records

Nothing is deleted. Two columns mark the rows to leave out of the graph:
- `is_duplicate`: another row has the same title (ignoring case and punctuation). The copy with the longest abstract is kept, the others are marked.
- `is_review_record`: decision letters, author responses, comments, replies, and titles that are only "References" or "Book Review".

Only identical titles are caught, so a preprint whose title was reworded for publication is not.

In [10]:
title_key = (df["title"].fillna("").str.lower()
             .str.replace(r"[^\w\s]", "", regex=True)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())

# within each title the copy with the longest abstract comes first and is kept
order = pd.DataFrame({"key": title_key, "length": df["abstract"].fillna("").str.len()})
order = order.sort_values(["key", "length"], ascending=[True, False])
duplicate = order.duplicated("key", keep="first") & (order["key"] != "")
df["is_duplicate"] = duplicate.reindex(df.index)

t = df["title"].fillna("").str.lower().str.strip()
df["is_review_record"] = (t.str.match(r"(author response|comment on|decision letter|reply to|review for)\b")
                          | t.isin(["references", "book review"]))

print(df["is_duplicate"].sum(), "rows marked as duplicates")
print(df["is_review_record"].sum(), "peer-review records / junk titles")
df.loc[df["is_duplicate"] | df["is_review_record"], ["title", "year", "doi"]].head(15)

29 rows marked as duplicates
18 peer-review records / junk titles


,title,year,doi
56,Artificial light at night (ALAN) disrupts timing of floral resource availability,2026,https://doi.org/10.5167/uzh-434892
59,Artificial light at night outweighs temperature in lengthening urban growing seasons,2025,https://doi.org/10.17169/refubium-51076
87,Impact of artificial light at night and night shift work on brain functions and metabolism,2025,https://doi.org/10.34734/fzj-2025-04485
121,ALANizer: Design and validation of experimental lighting rig for studying artificial light at ni...,2025,https://doi.org/10.1101/2025.08.30.673291
136,"Bright nights, dim futures: Artificial light at night disrupts behavioral patterns in animals: A...",2026,https://doi.org/10.5281/zenodo.20607738
174,"Review for ""The influence of artificial light at night on algal phenol concentrations can mediat...",2025,https://doi.org/10.1098/rspb.2025.1600/v2/review1
183,"Review for ""The influence of artificial light at night on algal phenol concentrations can mediat...",2025,https://doi.org/10.1098/rspb.2025.1600/v1/review2
187,"Review for ""The influence of artificial light at night on algal phenol concentrations can mediat...",2024,https://doi.org/10.1098/rspb.2025.1600/v1/review1
194,"Review for ""Sexually dimorphic and clock gene-specific effects of artificial light at night on D...",2025,https://doi.org/10.1098/rspb.2025.1170/v1/review2
203,"Review for ""Sexually dimorphic and clock gene-specific effects of artificial light at night on D...",2025,https://doi.org/10.1098/rspb.2025.1170/v1/review1


## 7. Save the table

Run this after the scope and marker steps so the saved CSV includes the `human_health`, `scope`, `is_duplicate` and `is_review_record` columns.

In [11]:
df.to_csv("../data/output/table_abstracts.csv", index=False)


## 8. Link matched words to KG items

`kg/term_map.csv` says which KG item each word stands for (for example "sky brightness" is the item `night sky brightness`), and whether the match is `exact` or `broad`. Rules:
- Quantities, units and methods use the map.
- A method or discipline word with no map row becomes an item of its own.
- The discipline rows in `term_map.csv` are not used: a field (astronomy) is not the same as the people who work in it (astronomers), and the curated `used_by` statements already cover people.
- Other matches (intensity, wavelength, species, ...) make no statement for now.

Papers used: in scope, not a duplicate, not a peer-review record, and with a DOI (the DOI is the paper's ID in the graph).

In [12]:
term_map = pd.read_csv("../kg/term_map.csv")
column_to_category = {"methods": "method", "quantities": "quantity",
                      "units": "unit", "disciplines": "discipline"}
lookup = {(r.category, r.term): (r.item, r.match) for r in term_map.itertuples()}

keep = df[(df["scope"] == "in scope") & ~df["is_duplicate"] & ~df["is_review_record"]]
print(len(keep), "papers in scope, not duplicate or review record;",
      keep["doi"].isna().sum(), "without a DOI are dropped")
keep = keep[keep["doi"].notna()]

rows = []
for r in keep.itertuples():
    for column, category in column_to_category.items():
        for term in getattr(r, column).split("; "):
            if not term:
                continue
            if category in ("quantity", "unit", "method") and (category, term) in lookup:
                item, match = lookup[(category, term)]
                item_class = "method" if category == "method" else None
            elif category in ("method", "discipline"):
                item, match, item_class = term, "exact", category
            else:
                continue
            rows.append({"paper": r.doi, "year": r.year, "term": term, "item": item,
                         "match": match, "item_class": item_class})

mentions = pd.DataFrame(rows).drop_duplicates()
mentions.to_csv("../data/output/mentions.csv", index=False)

print(len(mentions), "mention statements from", mentions["paper"].nunique(),
      "of", len(keep), "papers")
print(mentions["match"].value_counts())
print(mentions["item"].value_counts().head(15))
mapped = term_map.loc[term_map["category"] != "discipline", "term"]
print("map terms never found:", sorted(set(mapped) - set(mentions["term"])))

1634 papers in scope, not duplicate or review record; 109 without a DOI are dropped
1069 mention statements from 668 of 1525 papers
match
exact    1058
broad      11
Name: count, dtype: int64
item
conservation            232
regulation              106
ecology                  99
lx                       98
satellite                68
radiance                 43
night sky                41
urban planning           37
remote sensing           36
illuminance              35
night sky brightness     25
skyglow                  22
tourism                  22
lighting design          21
radiometer               20
Name: count, dtype: int64
map terms never found: ['PAR', 'W/m2', 'cd/m2', 'magnitudes per square arcsecond', 'spectral irradiance', 'spectral radiance', 'µmol/s', 'μmol photons']


## 9. A small local graph (toy model)

Two separate graphs: `seed` holds the curated statements (seed triples and item classes), `mentioned` holds the statements mined from papers. They are only combined into `g` for querying, so mined links never get mixed into the curated ones.

A broad match uses the property `mentions_broadly` instead of `mentions`, so it can be left out of a query.

In [13]:
from urllib.parse import quote
from rdflib import Graph, Namespace, Literal, URIRef, RDF, RDFS, XSD

EX = Namespace("http://example.org/alan/")


def node(text):
    return EX[quote(str(text).strip().replace(" ", "_"), safe="_")]


# curated graph: seed statements and item classes
seed = Graph()
for _, r in pd.read_csv("../kg/seed_statements.csv").iterrows():
    for name in (r["subject"], r["object"]):
        seed.add((node(name), RDFS.label, Literal(name)))
    seed.add((node(r["subject"]), EX[r["property"]], node(r["object"])))
for _, r in pd.read_csv("../kg/items.csv").iterrows():
    seed.add((node(r["item"]), RDF.type, node(r["class"])))

# mined graph: "paper mentions item"
mentioned = Graph()
for r in mentions.itertuples():
    paper = URIRef(r.paper)
    if pd.notna(r.year):
        mentioned.add((paper, EX.year, Literal(int(r.year), datatype=XSD.integer)))
    mentioned.add((node(r.item), RDFS.label, Literal(r.item)))
    if isinstance(r.item_class, str):
        mentioned.add((node(r.item), RDF.type, EX[r.item_class]))
    predicate = EX.mentions if r.match == "exact" else EX.mentions_broadly
    mentioned.add((paper, predicate, node(r.item)))

g = seed + mentioned
print(len(seed), "curated triples;", len(mentioned), "mention triples;", len(g), "in total")

78 curated triples; 1796 mention triples; 1863 in total


## 10. Two questions from `questions.md`

1. Which measurement methods are mentioned in ALAN papers, and in how many papers?
2. Which disciplines appear together with which methods? (Papers that mention both words: this shows co-occurrence, not that a discipline used a method.)

In [14]:
PREFIXES = """PREFIX ex: <http://example.org/alan/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

q1 = PREFIXES + """
SELECT ?label (COUNT(DISTINCT ?paper) AS ?papers) WHERE {
  ?paper ex:mentions|ex:mentions_broadly ?m .
  ?m a ?cls ; rdfs:label ?label .
  FILTER(?cls IN (ex:method, ex:instrument))
} GROUP BY ?label ORDER BY DESC(?papers)
"""
print("Question 1: methods mentioned, and in how many papers")
for row in g.query(q1):
    print("  ", row.label, row.papers)

q2 = PREFIXES + """
SELECT ?d ?m (COUNT(DISTINCT ?paper) AS ?papers) WHERE {
  ?paper ex:mentions ?di , ?mi .
  ?di a ex:discipline ; rdfs:label ?d .
  ?mi a ?cls ; rdfs:label ?m .
  FILTER(?cls IN (ex:method, ex:instrument))
} GROUP BY ?d ?m ORDER BY DESC(?papers) LIMIT 15
"""
print()
print("Question 2: disciplines and methods in the same paper")
for row in g.query(q2):
    print("  ", row.d, "+", row.m, row.papers)

Question 1: methods mentioned, and in how many papers
   satellite 68
   remote sensing 36
   radiometer 20
   Sky Quality Meter 10
   multispectral imaging device 8
   hyperspectral 6
   drone 5
   all-sky 4
   digital camera 4
   dose-response 4
   photometer 4
   International Space Station 2
   lux meter 2

Question 2: disciplines and methods in the same paper
   conservation + satellite 11
   conservation + remote sensing 10
   night sky + Sky Quality Meter 7
   night sky + satellite 7
   tourism + satellite 5
   night sky + all-sky 4
   skyglow + satellite 4
   tourism + radiometer 3
   tourism + remote sensing 3
   ecology + satellite 3
   urban planning + satellite 3
   night sky + radiometer 3
   regulation + satellite 3
   conservation + Sky Quality Meter 2
   ecology + hyperspectral 2


## Appendix A: seed KG overview (context only)

Not part of the mining flow. Shows how the 40 hand-entered triples in `kg/seed_statements.csv` map onto items, and which items are classified in `kg/items.csv`.

In [15]:
kg = pd.read_csv("../kg/seed_statements.csv")

items = pd.DataFrame({"item": sorted(set(kg["subject"]) | set(kg["object"]))})

def roles(name):
    as_subject = sorted(set(kg.loc[kg["subject"] == name, "property"]))
    as_object = sorted(set(kg.loc[kg["object"] == name, "property"]))
    return pd.Series({"subject_of": ", ".join(as_subject), "object_of": ", ".join(as_object)})

items = pd.concat([items, items["item"].apply(roles)], axis=1)
print(len(items))
items

27


,item,subject_of,object_of
0,V-band,,given_in_band
1,W/m2,,reported_in
2,W/m2/nm,,reported_in
3,W/m2/sr,,reported_in
4,W/m2/sr/nm,,reported_in
5,astronomers,,used_by
6,biologists,,used_by
7,cd/m2,,reported_in
8,ecologists,,used_by
9,illuminance,"photometric_version_of, reported_in, used_by",can_derive


In [16]:
items_cls = pd.read_csv("../kg/items.csv")
print(items_cls.shape)
print(items_cls["class"].value_counts())
known = set(kg["subject"]) | set(kg["object"])
print("not found in seed_statements:", sorted(set(items_cls["item"]) - known))



(11, 2)
class
quantity         8
spectral band    2
instrument       1
Name: count, dtype: int64
not found in seed_statements: []


In [17]:
g.serialize(destination="../data/output/alan_toy.ttl", format="turtle")
print("saved", len(g), "triples")

saved 1863 triples


## Appendix B: checks used while extending the word lists

Run after section 4. They show what the in-scope papers with no match are about, and whether a map term is rare or not matching.

In [18]:
print(df.columns.tolist())

['first_author', 'year', 'venue', 'title', 'doi', 'is_open_access', 'abstract', 'text', 'methods', 'quantities', 'units', 'disciplines', 'species', 'human_health', 'scope', 'is_duplicate', 'is_review_record']


In [19]:
target = df[(df["scope"] == "in scope") & (df[list(categories)] == "").all(axis=1)]
print(len(target), "in-scope papers with no match")

for word in ["species", "circadian"]:
    hits = target[target["text"].str.contains(word, case=False)]
    print(word, "-", len(hits), "papers")
    for t in hits["title"].sample(min(8, len(hits)), random_state=1):
        print("  -", t)
    print()

277 in-scope papers with no match
species - 34 papers
  - Spatiotemporal dynamics and driving factors of Dioryctria in China
  - Anthropogenic disturbance and mixed-species groups
  - Herbivory rate is elevated but orb-weaver spider growth unaffected by artificial light at night in subtropical forest
  - Dispersal flights of termites in an urban ecosystem in Northeastern Brazil
  - How developmental stressors affect mitochondrial respiratory function: a systematic review and meta‐analysis
  - Evidence mapping of the causes and effects of habitat change attributed to offshore wind development is important for determining impacts on fisheries species
  - Effects of complex soundscapes on the occurrence of Anaxipha pallidula in isolated green spaces in Tokyo
  - Phenotypic Flexibility in the City: A Meta‐Analysis on Variation

circadian - 21 papers
  - The Circadian Rhythm of Asthma Immune Metabolism
  - Living in the wrong schedule: the particular Chilean circadian dilemma and its impact

In [20]:
for t in target["title"].sample(30, random_state=7):
    print("-", t)

- Correction: Partial night lighting may reduce the physiological impact of artificial light at night on captive zebra finches
- Distribución espacio-temporal del peso al nacer en Argentina y su relación con las disparidades socioeconómicas y ambientales
- Coding the Circadian: The Rise of Anti-Jet Lag Technology
- High-resolution population density mapping in urban areas using a contextualized geographically weighted neural network (CGWNN) model
- Could Neuroscience and Smart Technologies Be the Answer to Inclusive Street Lighting?
- The Circadian Rhythm
- PUBLIC OPINION OF LIGHTING IN SLOTTSSKOGEN AND SUGGESTIONS FOR FUTURE CHANGES
- Seascape connectivity: evidence, knowledge gaps and implications for temperate coastal ecosystem restoration practice and policy
- Impacts of urban stressors on key aquatic organisms : biofilms and thinlip mullet
- Light Pollution: A Silent but Solvable Threat to Earth’s Ecological Balance
- Consensus-standpunten Light for Public Health Initiative
- Envi

In [21]:
c = df[df["text"].str.contains("cancer", case=False) & (df["scope"] == "in scope")]
print(len(c), "in-scope papers mention cancer")
for t in c["title"].head(15):
    print("-", t)

17 in-scope papers mention cancer
- Impact of artificial light at night and night shift work on brain functions and metabolism
- Chronic Artificial Light at Night Exposure Disrupts Circadian Rhythms and Modulates P53 Gene Expression in a Rat Model of Colorectal Cancer
- Light pollution and its impact on human health and wildlife
- Hepatocellular Carcinoma Around the Clock
- Editorial: ChronoBiophotonics
- Light pollution: environmental and health impacts at high latitudes
- Associations of light at night exposure and environmental pollutants with risk for thyroid cancer in Fujian, China: a spatio-temporal analysis
- Exploring the Role of Artificial Light and Tanning for Skin Cancer
- Exploring the Role of Artificial Light and Tanning for Skin Cancer
- The molecular interplay between the gut microbiome and circadian rhythms: an integrated review
- Pluripotency and Transcriptional Pausing Disrupt Circadian Rhythms to Facilitate the Enrichment of Cancer Stem Cells
- Collegium Ramazzini St

In [22]:
for term in ["cd/m2", "W/m2", "PAR", "spectral irradiance", "spectral radiance"]:
    n_cols = df[["quantities", "units"]].apply(lambda c: c.str.contains(term, regex=False)).any(axis=1).sum()
    n_text = df["text"].str.contains(term, regex=False).sum()
    print(term, "- in columns:", n_cols, "| literally in text:", n_text)

cd/m2 - in columns: 0 | literally in text: 0
W/m2 - in columns: 1 | literally in text: 2
PAR - in columns: 0 | literally in text: 8
spectral irradiance - in columns: 1 | literally in text: 1
spectral radiance - in columns: 0 | literally in text: 0


In [23]:
checks = {"PAR (whole word)": r"\bPAR\b",
          "cd/m2 variants": r"cd\s*[/ ]?\s*m[-−²2]",
          "candela": r"(?i)candela",
          "W/m2 variants": r"W\s*[/ ]\s*m[-−²2]"}
for name, pat in checks.items():
    hits = df[df["text"].str.contains(pat, regex=True)]
    print(name, "-", len(hits), "papers")
    for t in hits["title"].head(3):
        print("   ", t[:90])

PAR (whole word) - 0 papers
cd/m2 variants - 2 papers
    BIO 340 Final Project: Effects of Light Pollution on Bat Activity
    Assessment of Light Pollution at New Crescent Moon Observation Sites in Terengganu, Malays
candela - 0 papers
W/m2 variants - 4 papers
    Does outdoor light contribute to ovarian cancer mortality? Insights from a case-control st
    Light Wavelength Modulates the Effects of Lighted Nights on Sleep, Metabolism and Oxidativ
    A multi-level optimization and evaluation framework for electro-optical media facade light


## Appendix C: scratch snippets (commented out)

Uncomment to use. The first shows a random sample of papers that matched nothing; the second prints full abstracts for chosen row numbers.

In [24]:
# df[no_match][["title", "abstract"]].sample(10, random_state=1)

In [25]:
# for i in [144, 1716, 940]:
#     print(i, df.loc[i, "title"])
#     print(df.loc[i, "abstract"])
#     print()